# Modelo B — fine-tuning do focinho com ArcFace (open-set)

Fecha o gap de **verificação** da baseline congelada (FAR@FRR=1% ~54%).
A baseline usava um embedding genérico (ImageNet, nunca viu bois): ordena bem
(Rank-1 98%) mas não *afasta* genuínos de impostores. Metric learning (ArcFace)
treina exatamente isso → abre um vale entre as distribuições → FAR despenca.

**Protocolo honesto (regra de ouro do projeto):**
- Split **por animal**: os bois de teste **nunca** aparecem no treino (open-set real).
- Treina só nas identidades de treino; avalia identificação + verificação nas
  identidades de teste, nunca vistas.
- Compara fine-tuned **vs** baseline congelada **no mesmo split de teste** —
  a melhora tem que ser honesta.

Rode em GPU (Colab: Ambiente de execução → Alterar tipo → GPU T4).


## 1. Setup

In [ ]:
import torch, torchvision, sys
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA (troque para runtime GPU!)')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


## 2. Baixar o dataset de focinho (Zenodo 6324361, ~614 MB)

In [ ]:
import os, pathlib, urllib.request, zipfile
ZIP = pathlib.Path('BeefCattle_Muzzle_database.zip')
ROOT = pathlib.Path('BeefCattle_Muzzle_Individualized')
if not ROOT.exists():
    if not ZIP.exists():
        url = 'https://zenodo.org/api/records/6324361/files/BeefCattle_Muzzle_database.zip/content'
        print('baixando...'); urllib.request.urlretrieve(url, ZIP)
    print('extraindo...')
    with zipfile.ZipFile(ZIP) as z: z.extractall('.')
folders = sorted(p for p in ROOT.iterdir() if p.is_dir() and not p.name.startswith('._'))
print(len(folders), 'animais')


## 3. Split POR ANIMAL (open-set)

Identidades de treino e de teste são disjuntas. Os bois de teste são novos —
é assim que se mede se o embedding generaliza para animais nunca vistos.

In [ ]:
import numpy as np, random
IMG_EXT = {'.jpg','.jpeg','.png','.bmp'}
def imgs_of(folder):
    return [q for q in sorted(folder.rglob('*')) if q.suffix.lower() in IMG_EXT and not q.name.startswith('._')]

animals = [(f.name, imgs_of(f)) for f in folders]
animals = [(n,ps) for n,ps in animals if len(ps) >= 4]     # precisa de fotos p/ galeria+probe
rng = random.Random(42); rng.shuffle(animals)

n_test = max(1, int(0.25 * len(animals)))
test_animals  = animals[:n_test]
train_animals = animals[n_test:]
print(f'treino: {len(train_animals)} animais | teste (nunca vistos): {len(test_animals)} animais')

train_items, train_label = [], {}
for i,(name,ps) in enumerate(train_animals):
    train_label[name] = i
    for p in ps: train_items.append((p, i))
N_CLASSES = len(train_animals)
print(len(train_items), 'imagens de treino |', N_CLASSES, 'classes')


## 4. Dataset + augmentations

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

TRAIN_TF = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3,0.3,0.2,0.02),   # luz/umidade do focinho variam
    transforms.RandomRotation(12),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
EVAL_TF = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class MuzzleDS(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), y

train_loader = DataLoader(MuzzleDS(train_items, TRAIN_TF), batch_size=32,
                          shuffle=True, num_workers=2, drop_last=True)


## 5. Modelo: ResNet50 + cabeça de embedding + ArcFace

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torchvision import models

class MuzzleNet(nn.Module):
    def __init__(self, emb_dim=512):
        super().__init__()
        bb = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        bb.fc = nn.Linear(2048, emb_dim)
        self.backbone = bb
        self.bn = nn.BatchNorm1d(emb_dim)
    def forward(self, x):
        return self.bn(self.backbone(x))       # embedding (nao-normalizado)

class ArcFace(nn.Module):
    def __init__(self, in_f, n_cls, s=30.0, m=0.50):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_cls, in_f)); nn.init.xavier_uniform_(self.W)
        self.s, self.m = s, m
    def forward(self, emb, y):
        x = F.normalize(emb); W = F.normalize(self.W)
        cos = (x @ W.t()).clamp(-1+1e-7, 1-1e-7)
        theta = torch.acos(cos)
        target = torch.cos(theta + self.m)
        oh = F.one_hot(y, cos.size(1)).float()
        return self.s * (oh*target + (1-oh)*cos)

net  = MuzzleNet(512).to(DEVICE)
head = ArcFace(512, N_CLASSES).to(DEVICE)


## 6. Treino

In [ ]:
import torch.optim as optim
EPOCHS = 12
opt = optim.AdamW([
    {'params': net.backbone.parameters(),  'lr': 1e-4},
    {'params': net.bn.parameters(),        'lr': 1e-3},
    {'params': head.parameters(),          'lr': 1e-3},
], weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
crit = nn.CrossEntropyLoss()

for ep in range(EPOCHS):
    net.train(); head.train(); tot=0; correct=0; loss_sum=0
    for x,y in train_loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits = head(net(x), y)
        loss = crit(logits, y)
        loss.backward(); opt.step()
        loss_sum += loss.item()*len(y); tot += len(y)
        correct += (logits.argmax(1)==y).sum().item()
    sched.step()
    print(f'época {ep+1}/{EPOCHS}  loss={loss_sum/tot:.3f}  acc_treino={correct/tot:.3f}')
torch.save(net.state_dict(), 'muzzlenet_arcface.pt')
print('modelo salvo em muzzlenet_arcface.pt')


## 7. Avaliação open-set (bois de TESTE, nunca vistos)

Métricas iguais às do repo: identificação (Rank-1/mAP) e verificação
(EER, FAR@FRR=1%). Compara o modelo fine-tuned contra a baseline congelada.

In [ ]:
@torch.no_grad()
def embed_all(model, items):
    model.eval(); vecs=[]; labs=[]
    dl = DataLoader(MuzzleDS(items, EVAL_TF), batch_size=64, num_workers=2)
    for x,y in dl:
        v = model(x.to(DEVICE)).cpu().numpy()
        vecs.append(v); labs.append(y.numpy())
    emb = np.vstack(vecs).astype('float32')
    emb /= (np.linalg.norm(emb,axis=1,keepdims=True)+1e-8)
    return emb, np.concatenate(labs)

# itens de teste (rotulados por indice local de identidade de teste)
test_items=[]; tlab={}
for i,(name,ps) in enumerate(test_animals):
    tlab[name]=i
    for p in ps: test_items.append((p, i))

def identification(emb, labels, seed=42):
    rng=np.random.default_rng(seed); gal=[]; prb=[]
    for l in np.unique(labels):
        idx=np.where(labels==l)[0]; rng.shuffle(idx); k=len(idx)//2
        gal+=list(idx[:k]); prb+=list(idx[k:])
    gal,prb=np.array(gal),np.array(prb)
    sims=emb[prb]@emb[gal].T; order=np.argsort(-sims,axis=1)
    ranked=labels[gal][order]; corr=ranked==labels[prb][:,None]
    aps=[]
    for i in range(len(prb)):
        rel=corr[i]; n=rel.sum()
        if n==0: aps.append(0); continue
        cum=np.cumsum(rel); prec=cum/(np.arange(len(rel))+1)
        aps.append((prec*rel).sum()/n)
    return corr[:,0].mean(), corr[:,:5].any(1).mean(), float(np.mean(aps))

def verification(emb, labels, n=20000, seed=42):
    rng=np.random.default_rng(seed); by={l:np.where(labels==l)[0] for l in np.unique(labels)}
    multi=[l for l,ix in by.items() if len(ix)>=2]
    gen=[emb[a]@emb[b] for _ in range(n) for a,b in [rng.choice(by[multi[rng.integers(len(multi))]],2,replace=False)]]
    imp=[]
    while len(imp)<n:
        i,j=rng.integers(len(labels)),rng.integers(len(labels))
        if labels[i]!=labels[j]: imp.append(emb[i]@emb[j])
    gen,imp=np.array(gen),np.array(imp)
    ths=np.linspace(-1,1,400)
    far=np.array([(imp>=t).mean() for t in ths]); frr=np.array([(gen<t).mean() for t in ths])
    eer=float((far[np.argmin(np.abs(far-frr))]+frr[np.argmin(np.abs(far-frr))])/2)
    return eer, float(far[np.argmin(np.abs(frr-0.01))]), gen.mean(), imp.mean()

def report(tag, emb, labels):
    r1,r5,mAP = identification(emb,labels)
    eer,far,gm,im = verification(emb,labels)
    print(f'[{tag}] Rank-1={r1*100:5.1f}%  Rank-5={r5*100:5.1f}%  mAP={mAP*100:5.1f}%'
          f'  | EER={eer*100:4.1f}%  FAR@FRR1%={far*100:5.1f}%  (gen {gm:.2f} / imp {im:.2f})')

# baseline congelada (ImageNet puro) no MESMO split de teste
class Frozen(nn.Module):
    def __init__(self):
        super().__init__(); bb=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); bb.fc=nn.Identity(); self.bb=bb
    def forward(self,x): return self.bb(x)
fro = Frozen().to(DEVICE)

emb_fro, lab = embed_all(fro, test_items)
emb_ft,  _   = embed_all(net, test_items)
print('=== avaliação em', len(test_animals), 'animais NUNCA vistos ===')
report('congelada ', emb_fro, lab)
report('fine-tuned', emb_ft,  lab)


## 8. Leitura

- Espere **FAR@FRR=1% cair muito** (o número que importa pro caso de crédito).
  A identificação já era alta; o ganho principal é na separação genuíno/impostor.
- Se o FAR ainda estiver alto: mais épocas, `m` maior (0.5→0.6), mais augmentation,
  ou detectar/recortar o focinho antes (aqui as fotos já vêm recortadas).
- **Próximo passo real:** validar em focinho **Nelore** (dado do confinamento),
  não neste dataset norte-americano. Isto aqui prova a competência técnica.
